[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c10_tomb.ipynb)

# What Archimedes wanted on his grave: area and volume by slicing

In 75 BCE Cicero found Archimedes' lost tomb in the thorns at Syracuse. On it was a ball inside a can: the ball
fills exactly two-thirds of the can. Every area and volume formula here comes from the same three moves:
**slice it, write one slice, add them all**.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this cell first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, ToggleButtons

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the curve · slopes · slices and areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def strips(top, bottom, lo, hi, n):
    """Cut the region between two curves into n upright strips, each as tall as the region at its middle.
    Returns the strips' left edges, their width, their middles' heights, and the total area."""
    w = (hi - lo) / n
    left = lo + w * np.arange(n)
    mid = left + w / 2
    h = top(mid) - bottom(mid)
    return left, w, mid, float(np.sum(h * w))

print("Ready. Run the next cell.")

## 1 · Slice it, write one slice, add them all

Cut a shape into thin upright strips. Treat each strip as a rectangle: its height times its width. Add them all.
With more strips, the rectangles hug the curve better, and the total settles on the true area.

For a circle of radius 1, a strip at distance x from the centre reaches √(1 − x²) above the middle and the same
below (Pythagoras: x² + y² = 1). For an ellipse with half-widths a and b, every strip is b/a times as tall as the
circle's, so the area is πab.

In [ ]:
def slice_it(shape="circle", slices=4, b=3.0):
    a = 1.0 if shape == "circle" else 5.0
    h = 1.0 if shape == "circle" else b
    top = lambda x: h * np.sqrt(np.clip(1 - (x / a) ** 2, 0, None))
    bottom = lambda x: -top(x)
    left, w, mid, total = strips(top, bottom, -a, a, slices)
    true = math.pi * a * h
    fig, ax = plt.subplots(figsize=(9, 5.4))
    for k, (x0, m) in enumerate(zip(left, mid)):
        one = k == slices // 2
        ax.add_patch(plt.Rectangle((x0, bottom(m)), w, top(m) - bottom(m), facecolor=CORAL if one else LILAC,
                                   alpha=0.85 if one else 0.35, edgecolor=LILAC, lw=0.6 if slices < 60 else 0))
    xs = np.linspace(-a, a, 600)
    ax.plot(xs, top(xs), color=TEAL); ax.plot(xs, bottom(xs), color=TEAL)
    ax.set_aspect("equal"); ax.set_xlim(-a * 1.1, a * 1.1); ax.set_ylim(-max(a, h) * 1.1, max(a, h) * 1.1)
    name = "circle, radius 1" if shape == "circle" else f"ellipse, a = 5, b = {b:g}"
    ax.set_title(f"{name}:  {slices} strips add to {total:.5f}   ·   true area {true:.5f}")
    ax.set_xlabel("x")
    plt.show()
    if shape == "ellipse" and abs(b - a) < 1e-9: print("b = a: the oval is a circle again, area π × 5 × 5 = 25π.")

interact(slice_it, shape=ToggleButtons(options=["circle", "ellipse"], description="shape"),
         slices=IntSlider(value=4, min=4, max=200, step=4, description="slices"),
         b=FloatSlider(value=3.0, min=1.0, max=5.0, step=0.5, description="b (ellipse)"));

**Notice.** Four strips give about 3.26 for the circle. Drag the slices slider to 200: the total settles on
π = 3.14159… Then pick the ellipse: with a = 5 and b = 3 it is a circle of radius 5 squashed to three-fifths, so its
area is 25π × 3/5 = 15π ≈ 47.12. Drag b up to 5, and the oval turns back into a circle.

## 2 · Write one strip, then add them all exactly

The real trick is to write down one strip, any strip, as a formula. Then adding all of them is an integral:
**A = ∫ₐᵇ y dx**, "the area is the sum of every strip's height, times its tiny width, from the left edge to the
right edge". Here the computer adds them both ways: with many strips, and exactly.

In [ ]:
import sympy as sp
x, r, a_, b_ = sp.symbols("x r a b", positive=True)
circle = sp.integrate(2 * sp.sqrt(r**2 - x**2), (x, -r, r))
ellipse = 4 * sp.integrate(sp.Rational(3, 5) * sp.sqrt(25 - x**2), (x, 0, 5))
print("circle of radius r:            ", sp.simplify(circle))
print("x²/25 + y²/9 = 1 (4 quarters): ", sp.simplify(ellipse))
_, _, _, near = strips(lambda t: 3 / 5 * np.sqrt(25 - t**2), lambda t: 0 * t, 0, 5, 20000)
print("the same quarter with 20,000 strips, × 4:", round(4 * near, 4), " and 15π =", round(15 * math.pi, 4))

**Notice.** The textbook question "find the area enclosed by x²/25 + y²/9 = 1" is the circle of radius 5 with every
strip squashed to three-fifths: 25π × 3/5 = 15π.

## 3 · The dome and the can (Zu Geng's rule; beyond CBSE)

Zu Geng's rule (Europe calls it Cavalieri's principle): two solids of the same height whose slices match in area at
every height have the same volume. Take half a ball of radius r, a dome. Beside it, a can of radius r and height r with
a cone scooped out of it, point down. Slice both at height y. The dome's slice is a disc of area π(r² − y²). The
can's slice is a ring: πr² − πy². Equal at every height.

In [ ]:
def dome_and_can(y=1.5):
    R = 3.0
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11, 4.6))
    t = np.linspace(0, np.pi, 300)
    ax.plot(R * np.cos(t), R * np.sin(t), color=TEAL); ax.plot([-R, R], [0, 0], color=TEAL)
    s = math.sqrt(R * R - y * y)
    ax.plot([-s, s], [y, y], color=LILAC, lw=6)
    ax.set_title(f"dome: disc of radius {s:.2f}"); ax.set_aspect("equal"); ax.set_ylim(-0.3, R + 0.4)
    bx.plot([-R, -R, R, R], [0, R, R, 0], color=TEAL); bx.plot([-R, 0, R], [R, 0, R], color=DIM, ls="--")
    bx.plot([-R, -y], [y, y], color=LILAC, lw=6); bx.plot([y, R], [y, y], color=LILAC, lw=6)
    bx.set_title(f"can minus cone: ring from {y:.2f} to {R:.0f}"); bx.set_aspect("equal"); bx.set_ylim(-0.3, R + 0.4)
    disc, ring = math.pi * (R * R - y * y), math.pi * R * R - math.pi * y * y
    fig.suptitle(f"height y = {y:.2f}:   disc {disc:.3f}   ring {ring:.3f}", color=CORAL)
    plt.tight_layout(); plt.show()

interact(dome_and_can, y=FloatSlider(value=1.5, min=0.0, max=3.0, step=0.1, description="height y"));

**Notice.** The two numbers stay equal wherever the plane is. So the dome holds the same as the can minus its cone:
a cone is a third of its can, so the dome is two-thirds of it. Two domes make the ball: still two-thirds. That is
the carving on Archimedes' tomb.

In [ ]:
R = 3
ball, can = sp.Rational(4, 3) * sp.pi * R**3, sp.pi * R**2 * (2 * R)
print("ball:", ball, " can:", can, " ball ÷ can =", ball / can)
y = sp.symbols("y")
print("V = ∫ π y² dx for the ball:", sp.integrate(sp.pi * (r**2 - x**2), (x, -r, r)))

## 4 · The parabola's box

The region bounded by y² = 4ax and x = a: a strip at x runs from −2√(ax) to +2√(ax), so it is 4√(ax) tall. Adding
from 0 to a gives (8/3)a². Its box is 4a tall and a wide, 4a²: the parabola fills two-thirds of its box.

In [ ]:
A = 3
area = sp.integrate(4 * sp.sqrt(A * x), (x, 0, A))
print("a = 3:  area =", area, " box =", 4 * A * A, " fraction =", area / (4 * A * A))
print("top half only (the working that loses a mark):", sp.integrate(2 * sp.sqrt(A * x), (x, 0, A)))

## Sources
Cicero, *Tusculan Disputations* V.64–66; Plutarch, *Life of Marcellus* 17; MacTutor History of Mathematics:
Archimedes, Zu Geng, Zu Chongzhi, Bonaventura Cavalieri; Euclid, *Elements* XII.10. CBSE Mathematics syllabus
2026–27 (Application of Integrals). All retrieved 2026-10-24.